<a href="https://colab.research.google.com/github/NicolasRodrigues07/Space_connect_PromptAI/blob/main/projeto_aurora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Aurora — Mission Control AI

Sistema inteligente de monitoramento de missão espacial com cápsulas adaptativas e plantas-sensoras.

> Baseado no conceito do Projeto Aurora: cápsulas com plantas e sensores enviadas antes dos astronautas para aprender as melhores condições de vida no local.


## Célula 1 — Instalando dependências

Execute esta célula primeiro. Ela instala o Ollama e baixa o modelo Llama 3.2 1B.

In [2]:
#Instalar a dependência de descompactação (zstd) e o Ollama no sistema
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

#Instalar a biblioteca Python do Ollama
%pip install ollama -q

import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguarda 5 segundos para o servidor estabilizar no Colab
time.sleep(5)
import ollama
ollama.pull('llama3.2:1b')

print("\nINSTALADO COM SUCESSO! O Projeto Aurora está pronto para decolar.")

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

## Célula 2 — Importações e configuração

In [4]:
import ollama
import random
import time

# Configuração do sistema de alertas

LIMITES = {
    # Parâmetro          : (min_normal, max_normal)
    "temperatura_modulo" : (15, 35),    # °C
    "bateria"            : (25, 100),   # %
    "geracao_solar"      : (60, 100),   # %
    "radiacao"           : (0, 50),     # mSv/h
    "umidade_capsula"    : (40, 70),    # %
    "bpm_astronauta"     : (50, 100),   # BPM (batimentos cardíacos)
}

COMUNICACAO_STATUS = ["estável", "estável", "estável", "instável", "sem sinal"]

print("Configuração carregada com sucesso!")
print(f"   Monitorando {len(LIMITES)} parâmetros + status de comunicação")

Configuração carregada com sucesso!
   Monitorando 6 parâmetros + status de comunicação


## Célula 3 — Simulação de dados da missão

In [6]:
def gerar_dados_missao(modo="normal"):
    """
    Gera dados simulados da cápsula Aurora.
    modo='normal'  -> parâmetros saudáveis
    modo='critico' -> parâmetros fora do limite
    """
    if modo == "normal":
        return {
            "temperatura_modulo" : round(random.uniform(18, 32), 1),
            "bateria"            : round(random.uniform(40, 95), 1),
            "geracao_solar"      : round(random.uniform(65, 98), 1),
            "radiacao"           : round(random.uniform(5, 40), 2),
            "umidade_capsula"    : round(random.uniform(45, 65), 1),
            "bpm_astronauta"     : random.randint(58, 90),
            "comunicacao"        : random.choice(["estável", "estável", "estável"]),
            "plantas_status"     : random.choice(["saudáveis", "saudáveis", "levemente estressadas"]),
        }
    else:  # modo critico
        return {
            "temperatura_modulo" : round(random.uniform(48, 60), 1),
            "bateria"            : round(random.uniform(5, 18), 1),
            "geracao_solar"      : round(random.uniform(15, 35), 1),
            "radiacao"           : round(random.uniform(75, 120), 2),
            "umidade_capsula"    : round(random.uniform(10, 25), 1),
            "bpm_astronauta"     : random.randint(115, 145),
            "comunicacao"        : random.choice(["instável", "sem sinal"]),
            "plantas_status"     : random.choice(["estressadas", "murchando"]),
        }

print("Função de simulação pronta!")
print("   Use gerar_dados_missao('normal') ou gerar_dados_missao('critico')")

Função de simulação pronta!
   Use gerar_dados_missao('normal') ou gerar_dados_missao('critico')


## Célula 4 — Lógica de alertas e tomada de decisão

In [7]:
def verificar_alertas(dados):
    """
    Verifica cada parâmetro contra os limites definidos.
    Retorna lista de alertas e nível de criticidade geral.
    """
    alertas = []

    for param, (minv, maxv) in LIMITES.items():
        valor = dados.get(param)
        if valor is None:
            continue
        if valor < minv or valor > maxv:
            alertas.append({
                "parametro": param,
                "valor": valor,
                "limite": f"{minv}–{maxv}",
                "tipo": "ALTO" if valor > maxv else "BAIXO"
            })

    if dados.get("comunicacao") in ["instável", "sem sinal"]:
        alertas.append({
            "parametro": "comunicacao",
            "valor": dados["comunicacao"],
            "limite": "estável",
            "tipo": "FALHA"
        })

    # Nível de criticidade
    if len(alertas) == 0:
        nivel = "NORMAL"
    elif len(alertas) <= 2:
        nivel = "ATENÇÃO"
    else:
        nivel = "CRÍTICO"

    return alertas, nivel


def tomada_de_decisao(dados, alertas):
    """
    Regras automáticas de resposta para situações críticas.
    Simula as ações que a cápsula Aurora executaria.
    """
    acoes = []

    if dados.get("bateria", 100) < 20:
        acoes.append("⚡ MODO ECONOMIA ATIVADO — sistemas não essenciais desligados")

    if dados.get("temperatura_modulo", 25) > 40:
        acoes.append("❄️  RESFRIAMENTO EMERGENCIAL ATIVADO — ventilação máxima")

    if dados.get("radiacao", 0) > 60:
        acoes.append("☢️  ESCUDO DE RADIAÇÃO REFORÇADO — alertar tripulação")

    if dados.get("bpm_astronauta", 70) > 110:
        acoes.append("💓 CÁPSULA ADAPTATIVA: ajustando iluminação e temperatura para relaxamento")

    if dados.get("comunicacao") == "sem sinal":
        acoes.append("📡 PROTOCOLO BACKUP: tentando reconexão via antena secundária")

    if dados.get("umidade_capsula", 55) < 30:
        acoes.append("💧 UMIDIFICADORES ATIVADOS — risco de desidratação das plantas")

    if not acoes:
        acoes.append("✅ Todos os sistemas operando normalmente. Nenhuma ação necessária.")

    return acoes


print(" Lógica de alertas e decisão pronta!")

 Lógica de alertas e decisão pronta!


## Célula 5 — Exibição do painel de status

In [8]:
COR_RESET  = "\033[0m"
COR_VERDE  = "\033[92m"
COR_AMARELO= "\033[93m"
COR_VERMELHO="\033[91m"
COR_CIANO  = "\033[96m"
COR_NEGRITO= "\033[1m"


def cor_nivel(nivel):
    return {"NORMAL": COR_VERDE, "ATENÇÃO": COR_AMARELO, "CRÍTICO": COR_VERMELHO}.get(nivel, COR_RESET)


def exibir_painel(dados, alertas, nivel, acoes, analise_ia=""):
    print(COR_NEGRITO + COR_CIANO)
    print("-" * 30)
    print("   🌿 PROJETO AURORA — MISSION CONTROL AI")
    print("-" * 30)
    print(COR_RESET)

    # Status geral
    cor = cor_nivel(nivel)
    print(f"  STATUS GERAL: {cor}{COR_NEGRITO}{nivel}{COR_RESET}")
    print()

    # Dados dos sensores
    print(COR_NEGRITO + "  ── DADOS DA CÁPSULA ──" + COR_RESET)
    print(f"  🌡️ Temperatura do módulo : {dados['temperatura_modulo']}°C")
    print(f"  ⚡ Bateria               : {dados['bateria']}%")
    print(f"  ☀️ Geração solar         : {dados['geracao_solar']}%")
    print(f"  ☢️ Radiação              : {dados['radiacao']} mSv/h")
    print(f"  💧 Umidade da cápsula    : {dados['umidade_capsula']}%")
    print(f"  💓 BPM astronauta        : {dados['bpm_astronauta']} bpm")
    print(f"  📡 Comunicação           : {dados['comunicacao']}")
    print(f"  🌿 Plantas               : {dados['plantas_status']}")
    print()

    # Alertas
    if alertas:
        print(COR_VERMELHO + COR_NEGRITO + "  ── ALERTAS DETECTADOS ──" + COR_RESET)
        for a in alertas:
            print(f"  ⚠️  [{a['tipo']}] {a['parametro'].upper()}: {a['valor']} (normal: {a['limite']})")
        print()

    # Ações automáticas
    print(COR_NEGRITO + "  ── AÇÕES AUTOMÁTICAS ──" + COR_RESET)
    for acao in acoes:
        print(f"  {acao}")
    print()

    # Análise da IA
    if analise_ia:
        print(COR_CIANO + COR_NEGRITO + "  ── ANÁLISE DA IA (Llama) ──" + COR_RESET)
        # Quebra o texto da IA em linhas de até 50 chars para melhor leitura
        palavras = analise_ia.strip().split()
        linha = "  "
        for palavra in palavras:
            if len(linha) + len(palavra) > 52:
                print(linha)
                linha = "  " + palavra + " "
            else:
                linha += palavra + " "
        if linha.strip():
            print(linha)
        print()

    print(COR_CIANO + "-" * 30 + COR_RESET)


print("Função de exibição pronta!")

Função de exibição pronta!


## Célula 6 — Integração com a IA (Llama via Ollama)

In [9]:
def analisar_com_ia(dados, alertas, nivel):
    """
    Envia os dados da missão para o Llama e obtém uma análise inteligente.
    """
    alertas_str = "\n".join(
        [f"- {a['parametro']}: {a['valor']} (esperado: {a['limite']})" for a in alertas]
    ) if alertas else "Nenhum alerta."

    prompt_usuario = f"""Dados atuais da cápsula Aurora:
- Temperatura: {dados['temperatura_modulo']}°C
- Bateria: {dados['bateria']}%
- Geração solar: {dados['geracao_solar']}%
- Radiação: {dados['radiacao']} mSv/h
- Umidade da cápsula: {dados['umidade_capsula']}%
- BPM do astronauta: {dados['bpm_astronauta']}
- Comunicação: {dados['comunicacao']}
- Status das plantas: {dados['plantas_status']}

Nível de criticidade: {nivel}

Alertas ativos:
{alertas_str}

Faça uma análise breve da situação e recomende a próxima ação prioritária."""

    resposta = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Você é a inteligência artificial da cápsula espacial Aurora. "
                    "Sua missão é proteger os astronautas e as plantas-sensoras a bordo. "
                    "Analise os dados da missão e forneça recomendações diretas e objetivas. "
                    "Responda sempre em português, de forma concisa (máximo 3 frases)."
                )
            },
            {
                "role": "user",
                "content": prompt_usuario
            }
        ]
    )

    return resposta["message"]["content"]


print("Integração com Llama pronta!")

✅ Integração com Llama pronta!


## Célula 7 — EXECUTAR A MISSÃO

Mude `modo` para `"normal"` ou `"critico"` para simular diferentes situações.

In [10]:
#   ALTERE AQUI PARA TESTAR:
#   modo = "normal"
#   modo = "critico"

modo = "normal"   #mude aqui

print(f"Simulando missão no modo: {modo.upper()}...\n")

# 1. Gerar dados
dados = gerar_dados_missao(modo)

# 2. Verificar alertas
alertas, nivel = verificar_alertas(dados)

# 3. Tomar decisões automáticas
acoes = tomada_de_decisao(dados, alertas)

# 4. Analisar com IA
analise_ia = analisar_com_ia(dados, alertas, nivel)

# 5. Exibir painel
exibir_painel(dados, alertas, nivel, acoes, analise_ia)

Simulando missão no modo: NORMAL...


------------------------------
   🌿 PROJETO AURORA — MISSION CONTROL AI
------------------------------

  STATUS GERAL: NORMAL

  ── DADOS DA CÁPSULA ──
  🌡️ Temperatura do módulo : 31.0°C
  ⚡ Bateria               : 44.9%
  ☀️ Geração solar         : 71.4%
  ☢️ Radiação              : 10.45 mSv/h
  💧 Umidade da cápsula    : 61.6%
  💓 BPM astronauta        : 71 bpm
  📡 Comunicação           : estável
  🌿 Plantas               : saudáveis

  ── AÇÕES AUTOMÁTICAS ──
  ✅ Todos os sistemas operando normalmente. Nenhuma ação necessária.

  ── ANÁLISE DA IA (Llama) ──
  A análise atual da cápsula Aurora revela um nível 
  de criticidade NOVO, com o nível de radiação sendo 
  superior a 10 mSv/h. É fundamental aumentar a 
  vigilância e manter as comunicações em boas 
  condições para evitar qualquer dano adicional ao 
  equipamento ou aos seres vivos da cápsula. 

------------------------------


## Célula 8 (Nosso diferencial) — Monitoramento contínuo (3 ciclos)

In [11]:
from IPython.display import clear_output

CICLOS = 3       # quantos ciclos rodar
INTERVALO = 5    # segundos entre ciclos

modos_simulacao = ["normal", "normal", "critico"]  # alterna para mostrar alerta no 3º ciclo

for i in range(CICLOS):
    clear_output(wait=True)
    print(f"Ciclo de monitoramento {i+1}/{CICLOS}\n")

    modo_atual = modos_simulacao[i % len(modos_simulacao)]
    dados = gerar_dados_missao(modo_atual)
    alertas, nivel = verificar_alertas(dados)
    acoes = tomada_de_decisao(dados, alertas)

    analise_ia = analisar_com_ia(dados, alertas, nivel)
    exibir_painel(dados, alertas, nivel, acoes, analise_ia)

    if i < CICLOS - 1:
        print(f"\n Próxima leitura em {INTERVALO}s...")
        time.sleep(INTERVALO)

print("\nMonitoramento concluído.")

Ciclo de monitoramento 3/3


------------------------------
   🌿 PROJETO AURORA — MISSION CONTROL AI
------------------------------

  STATUS GERAL: CRÍTICO

  ── DADOS DA CÁPSULA ──
  🌡️ Temperatura do módulo : 54.8°C
  ⚡ Bateria               : 12.2%
  ☀️ Geração solar         : 23.0%
  ☢️ Radiação              : 96.55 mSv/h
  💧 Umidade da cápsula    : 18.1%
  💓 BPM astronauta        : 142 bpm
  📡 Comunicação           : instável
  🌿 Plantas               : murchando

  ── ALERTAS DETECTADOS ──
  ⚠️  [ALTO] TEMPERATURA_MODULO: 54.8 (normal: 15–35)
  ⚠️  [BAIXO] BATERIA: 12.2 (normal: 25–100)
  ⚠️  [BAIXO] GERACAO_SOLAR: 23.0 (normal: 60–100)
  ⚠️  [ALTO] RADIACAO: 96.55 (normal: 0–50)
  ⚠️  [BAIXO] UMIDADE_CAPSULA: 18.1 (normal: 40–70)
  ⚠️  [ALTO] BPM_ASTRONAUTA: 142 (normal: 50–100)
  ⚠️  [FALHA] COMUNICACAO: instável (normal: estável)

  ── AÇÕES AUTOMÁTICAS ──
  ⚡ MODO ECONOMIA ATIVADO — sistemas não essenciais desligados
  ❄️  RESFRIAMENTO EMERGENCIAL ATIVADO — ventilação máxima